# C4 — Notebook 06 (preliminary): Weather + Environment Alignment and Quality Assurance

**Purpose:** Join Notebook 01's **1,429** source-derived gauge-level supervised records with Notebook 04's ERA5-Land **`[14, 4]`** weather sequences and Notebook 05's **`[9]`** environmental vectors. This is **preliminary M6 data preparation**, not final three-modality fusion and not neural-network training.

**Safeguards:**
- **Read only** Notebook 01/04/05 sources, never touch Earth Engine or rerun spatial extraction.
- Match weather by `record_id + station_id + forecast_date + split`, environment by `station_id`.
- Cross-check polygon SHA256 across three modality contracts (candidate geometry unchanged).
- Keep **future-stage targets in a separate file** from all inputs; current stage is known historical input.
- Verify the simulated `T 09:00 Asia/Colombo` forecast cutoff; last historical weather hour `T 08:30`.
- Write new files only in `data/processed/c4/aligned/` after every required QA passes.

**Interpretation:** The original DMC source reports were nominally published at **09:30**, and historical ERA5-Land publication latency is not modeled. Therefore 09:00 is a **retrospective simulated hindcast origin**, not a demonstrated operational 09:00 warning capability. Candidate polygon QA remains provisional.

In [ ]:
# Cell 1 — Imports and explicit repository paths (no retrieval)
from __future__ import annotations
import hashlib, io, json, sys, tempfile, os
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

EXPECTED_N=1429
EXPECTED_GAUGES=39
EXPECTED_SPLITS={'TRAIN':809,'VALIDATION':269,'TEST':351}
EXPECTED_WEATHER_SHAPE=(1429,14,4)
EXPECTED_ENV_SHAPE=(1429,9)


def find_repo(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate/'c4_fusion'/'src'/'weather'/'load.py').is_file() and (candidate/'c4_fusion'/'src'/'environment'/'load.py').is_file():
            return candidate
    raise FileNotFoundError('Cannot find repo with both loaders. Run from J26-DS-314 or its c4_fusion/notebooks/ folder.')

ROOT=find_repo(Path.cwd())
MANIFESTS=ROOT/'data'/'processed'/'c4'/'manifests'
ENV_DIR=ROOT/'data'/'processed'/'c4'/'environment'
WEATHER_DIR=ROOT/'data'/'processed'/'c4'/'weather'
OUT_DIR=ROOT/'data'/'processed'/'c4'/'aligned'
INDEX_PATH=MANIFESTS/'c4_sample_index_1429.csv'
LABELS_PATH=MANIFESTS/'c4_training_labels_1429_PRIVATE.csv'
SPATIAL_QA_PATH=MANIFESTS/'c4_spatial_qa_39.csv'

for file in [INDEX_PATH,LABELS_PATH,SPATIAL_QA_PATH]:
    if not file.is_file():
        raise FileNotFoundError(f'Notebook 01 output missing: {file}. Run Notebook 01 Cell 9 first; do not recreate labels.')
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))

def sha256(path:Path)->str:
    h=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda:handle.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()

print('Repository:',ROOT)
print('Notebook 01 manifests:',MANIFESTS)
print('Notebook 04 weather:',WEATHER_DIR)
print('Notebook 05 environment:',ENV_DIR)
print('New, independent outputs:',OUT_DIR)
print('PASS: all required Notebook 01 manifests exist; no GEE requested.')

In [2]:
# Cell 2 — Load Notebook 01's target-free index and separate private supervised labels
index=pd.read_csv(INDEX_PATH,encoding='utf-8-sig',dtype={'record_id':'string','station_id':'string'},low_memory=False)
labels=pd.read_csv(LABELS_PATH,encoding='utf-8-sig',dtype={'record_id':'string','station_id':'string'},low_memory=False)
spatial=pd.read_csv(SPATIAL_QA_PATH,encoding='utf-8-sig',dtype={'station_id':'string'},low_memory=False)

require_index={'record_id','station_id','forecast_date','split','hindcast_origin_0900_local',
               'target_time_0900_local','polygon_repo_path','catchment_qa_status'}
require_labels={'record_id','station_id','forecast_date','split','stage_at_issue_m',
                'target_stage_next_day_m','target_delta_stage_m',
                'stage_0900_obs_time_local','target_0900_obs_time_local','source_unit'}
assert require_index.issubset(index.columns), f'Index missing {require_index-set(index.columns)}'
assert require_labels.issubset(labels.columns), f'Labels missing {require_labels-set(labels.columns)}'
assert {'station_id','polygon_sha256','catchment_qa_status','polygon_repo_path'}.issubset(spatial.columns)
assert len(index)==len(labels)==EXPECTED_N and len(spatial)==EXPECTED_GAUGES
assert index['record_id'].is_unique and labels['record_id'].is_unique and spatial['station_id'].is_unique
assert index[['station_id','forecast_date']].duplicated().sum()==0
assert labels[['station_id','forecast_date']].duplicated().sum()==0
assert index['station_id'].nunique()==labels['station_id'].nunique()==EXPECTED_GAUGES
assert index['record_id'].tolist()==labels['record_id'].tolist(), 'Notebook 01 index/label order drift'
assert (index['record_id'].astype(str)==index['station_id'].astype(str)+'_'+index['forecast_date'].astype(str)).all()
assert index['split'].value_counts().to_dict()==labels['split'].value_counts().to_dict()==EXPECTED_SPLITS
assert index['station_id'].isin(spatial['station_id']).all()
for col in ['record_id','station_id','forecast_date','split']:
    assert index[col].astype(str).equals(labels[col].astype(str)), f'Index vs labels mismatch: {col}'

# Future outcomes must NEVER occur in the target-free sample index.
BANNED_INPUT_FIELDS={'target_stage_next_day_m','target_delta_stage_m','stage_0900_next_day_m',
                     'stage_at_next_day_m','y_next_stage_m','y_delta_stage_m'}
assert not BANNED_INPUT_FIELDS.intersection(index.columns), 'Future numerical target exposed in feature index!'

forecast_dt=pd.to_datetime(index['forecast_date'],format='%Y-%m-%d',errors='raise')
assert forecast_dt.between('2025-11-17','2025-12-25').all()
periods={'TRAIN':('2025-11-17','2025-12-09'),
         'VALIDATION':('2025-12-10','2025-12-16'),
         'TEST':('2025-12-17','2025-12-25')}
for split,(first,last) in periods.items():
    assert forecast_dt[index['split'].eq(split)].between(first,last).all(),f'Unexpected {split} date'

print('PASS: Notebook 01 index and separate labels have 1429 identical ordered identities.')
print('PASS: 39 labelled gauges; 809/269/351 frozen chronological split.')
print('PASS: future target values absent from target-free index.')

PASS: Notebook 01 index and separate labels have 1429 identical ordered identities.
PASS: 39 labelled gauges; 809/269/351 frozen chronological split.
PASS: future target values absent from target-free index.


In [3]:
# Cell 3 — Load BOTH tested local Python loaders with full provenance verification
from c4_fusion.src.environment.load import EnvironmentFeatureLoader, FEATURE_COLUMNS as ENV_COLUMNS
from c4_fusion.src.weather.load import WeatherSequenceLoader, FEATURE_COLUMNS as WEATHER_COLUMNS

env_loader=EnvironmentFeatureLoader(repo_root=ROOT,verify_provenance=True,verify_polygon_files=False)
weather_loader=WeatherSequenceLoader(repo_root=ROOT,verify_provenance=True)
assert env_loader.station_count==40
assert weather_loader.shape==EXPECTED_WEATHER_SHAPE
assert len(ENV_COLUMNS)==9 and len(WEATHER_COLUMNS)==4

# Confirm all 39 labelled source polygons are the SAME versions in NB01, NB04, NB05.
spa=spatial.set_index('station_id')
env_manifest=env_loader.manifest
weather_manifest=weather_loader.manifest
assert env_manifest and weather_manifest
assert weather_manifest.get('target_free_index_sha256')==sha256(INDEX_PATH), (
    'Notebook 04 weather was extracted with a different Notebook 01 target-free index version. '
    'Keep the ORIGINAL c4_sample_index_1429.csv used by Notebook 04, or re-audit your weather extraction before aligning.')
assert set(index['station_id'])==set(weather_manifest['polygon_sha256'])
for sid in sorted(set(index['station_id'])):
    hashes={
        'Notebook 01':str(spa.at[sid,'polygon_sha256']),
        'Notebook 04':weather_manifest['polygon_sha256'].get(sid),
        'Notebook 05':env_manifest['station_polygon_sha256'].get(sid),
    }
    assert len(set(hashes.values()))==1,f'{sid}: polygon differs across notebooks: {hashes}'
    assert str(spa.at[sid,'catchment_qa_status'])==str(index.loc[index.station_id.eq(sid),'catchment_qa_status'].iloc[0]),f'Catchment QA flag changed: {sid}'
    assert str(spa.at[sid,'polygon_repo_path'])==str(index.loc[index.station_id.eq(sid),'polygon_repo_path'].iloc[0]),f'Catchment polygon path changed: {sid}'

print('PASS: Environment and weather loaders independently verified all saved source files.')
print('PASS: exact same candidate polygon SHA256 for 39 stations across NB01/NB04/NB05.')
print('Environment feature order:',list(ENV_COLUMNS))
print('Weather feature order:',list(WEATHER_COLUMNS))

PASS: Environment and weather loaders independently verified all saved source files.
PASS: exact same candidate polygon SHA256 for 39 stations across NB01/NB04/NB05.
Environment feature order: ['env_log1p_area_km2', 'env_mean_elevation_m', 'env_mean_slope_deg', 'env_tree_fraction', 'env_cropland_fraction', 'env_built_fraction', 'env_worldcover_water_fraction', 'env_jrc_persistent_water_fraction', 'env_hydrorivers_density_km_per_km2']
Weather feature order: ['precipitation_mm', 'temperature_C', 'dewpoint_C', 'surface_pressure_hPa']


In [4]:
# Cell 4 — Align and test ONE real station-day, then every record
# Weather uses record_id, station_id, forecast_date, split; environment uses station_id.
X_weather=weather_loader.for_station_days(index) # [1429,14,4] (copies)
X_environment=env_loader.for_station_days(index) # [1429,9] (copies)
assert X_weather.shape==EXPECTED_WEATHER_SHAPE and X_environment.shape==EXPECTED_ENV_SHAPE
assert X_weather.dtype==np.float32 and X_environment.dtype==np.float32
assert np.isfinite(X_weather).all() and np.isfinite(X_environment).all()

pilot_key='holombuwa_2025-11-17'
pilot_position=int(index.index[index['record_id'].eq(pilot_key)][0])
assert np.array_equal(X_weather[pilot_position],weather_loader.get_sequence(pilot_key))
assert np.array_equal(X_environment[pilot_position],env_loader.get_vector('holombuwa'))

# Environmental features must be invariant across station-days.
for sid,positions in index.groupby('station_id',sort=True).indices.items():
    assert np.array_equal(X_environment[positions],np.broadcast_to(env_loader.get_vector(sid),(len(positions),9))),f'{sid}: environment varied across dates'

print('PASS: One Holombuwa station-day correctly aligns weather [14,4] and environment [9].')
print('PASS: full weather',X_weather.shape,'+ environment',X_environment.shape,'aligned in Notebook 01 source order.')
print('PASS: environmental vectors stay fixed at each gauge, reused across dates.')
print('First sample:',index.loc[0,['record_id','forecast_date','split']].to_dict())
print('Holombuwa weather first/last day, four values:')
display(pd.DataFrame(X_weather[pilot_position],columns=list(WEATHER_COLUMNS),index=pd.date_range('2025-11-04',periods=14,freq='D').strftime('%Y-%m-%d')).round(4))
print('Holombuwa nine environmental values:')
display(pd.Series(X_environment[pilot_position],index=list(ENV_COLUMNS),name='value').round(5))

PASS: One Holombuwa station-day correctly aligns weather [14,4] and environment [9].
PASS: full weather (1429, 14, 4) + environment (1429, 9) aligned in Notebook 01 source order.
PASS: environmental vectors stay fixed at each gauge, reused across dates.
First sample: {'record_id': 'badalgama_2025-11-17', 'forecast_date': '2025-11-17', 'split': 'TRAIN'}
Holombuwa weather first/last day, four values:


,precipitation_mm,temperature_C,dewpoint_C,surface_pressure_hPa
2025-11-04,0.2034,24.365200,21.329901,976.276123
2025-11-05,2.0724,24.128700,21.678400,975.476013
2025-11-06,5.0872,24.016600,21.483900,974.739319
2025-11-07,8.3017,23.587000,21.936899,974.931396
2025-11-08,4.3774,23.787800,21.627001,975.600403
2025-11-09,6.5003,24.095100,21.739100,975.588623
2025-11-10,0.9254,24.409901,21.260500,975.057495
2025-11-11,8.6067,23.605600,21.764000,975.603516
2025-11-12,1.5150,24.261999,21.909800,975.281921
2025-11-13,12.7244,24.222200,22.195801,975.234070


Holombuwa nine environmental values:


env_log1p_area_km2                      5.04055
env_mean_elevation_m                  287.16037
env_mean_slope_deg                     16.83959
env_tree_fraction                       0.97924
env_cropland_fraction                   0.01395
env_built_fraction                      0.00215
env_worldcover_water_fraction           0.00000
env_jrc_persistent_water_fraction       0.00000
env_hydrorivers_density_km_per_km2      0.31067
Name: value, dtype: float32

In [5]:
# Cell 5 — Check observation cutoff, next-day target, units and intended split dates
# Parse the explicit Asia/Colombo local time strings without losing the original text.
def local_clock(series,purpose)->pd.Series:
    assert series.notna().all(),f'{purpose}: missing local timestamp'
    raw=series.astype(str)
    assert raw.str.endswith(' Asia/Colombo').all(), f'{purpose}: missing expected timezone'
    return pd.to_datetime(raw.str.removesuffix(' Asia/Colombo'),format='%Y-%m-%d %H:%M',errors='raise')

origin=local_clock(index['hindcast_origin_0900_local'],'simulated origin')
next_target_time=local_clock(index['target_time_0900_local'],'target time')
raw_current_time=local_clock(labels['stage_0900_obs_time_local'],'source current observation')
raw_target_time=local_clock(labels['target_0900_obs_time_local'],'source target observation')
assert origin.equals(raw_current_time) and next_target_time.equals(raw_target_time)
assert (origin==forecast_dt+pd.Timedelta(hours=9)).all()
assert (next_target_time==origin+pd.Timedelta(hours=24)).all()
assert labels['source_unit'].isin(['m','ft']).all()
assert set(labels.loc[labels.source_unit.eq('ft'),'station_id'])=={'nagalagam_street'}

# Weather index's most recent hour is T 03:00 UTC = T 08:30 local; 30min BEFORE origin.
wx_idx=weather_loader.index_table().set_index('record_id').loc[index['record_id']].reset_index()
assert (wx_idx['station_id'].astype(str).to_numpy()==index['station_id'].astype(str).to_numpy()).all()
assert wx_idx['forecast_date'].eq(index['forecast_date']).all()
assert wx_idx['split'].eq(index['split']).all()
assert wx_idx['weather_last_hour_local'].eq('08:30 Asia/Colombo').all()
assert wx_idx['last_weather_day'].eq(index['forecast_date']).all()
assert wx_idx['earliest_weather_day'].eq((forecast_dt-pd.Timedelta(days=13)).dt.strftime('%Y-%m-%d')).all()
last_hour_local=forecast_dt+pd.Timedelta(hours=8,minutes=30)
assert (origin-last_hour_local).eq(pd.Timedelta(minutes=30)).all()
# The weather loader independently checked source-day UTC windows and all 14x24 hourly records.

current=labels['stage_at_issue_m'].to_numpy(dtype=np.float32).reshape(-1,1)
y_next=labels['target_stage_next_day_m'].to_numpy(dtype=np.float32).reshape(-1,1)
y_delta=labels['target_delta_stage_m'].to_numpy(dtype=np.float32).reshape(-1,1)
assert np.isfinite(current).all() and np.isfinite(y_next).all() and np.isfinite(y_delta).all()
assert np.allclose(y_next-current,y_delta,atol=1e-4,rtol=1e-6)

# CRUCIAL: the future stage and delta are not in the model input tensors.
inputs={'X_weather':X_weather,'X_environment':X_environment,'stage_current_m':current}
targets={'y_next_stage_m':y_next,'y_delta_stage_m':y_delta}
assert not set(inputs).intersection(targets)
print('PASS: every weather last-hour at T 08:30 local precedes simulated T 09:00 origin by 30 minutes.')
print('PASS: every target is observed T+1 09:00; 24-hour observation-to-observation horizon.')
print('PASS: all stages converted to metres; original feet origin only Nagalagam Street.')
print('CAUTION: nominal public DMC report 09:30; 09:00 origin is SIMULATED retrospective hindcast.')

PASS: every weather last-hour at T 08:30 local precedes simulated T 09:00 origin by 30 minutes.
PASS: every target is observed T+1 09:00; 24-hour observation-to-observation horizon.
PASS: all stages converted to metres; original feet origin only Nagalagam Street.
CAUTION: nominal public DMC report 09:30; 09:00 origin is SIMULATED retrospective hindcast.


In [6]:
# Cell 6 — Numerical quality audit, split coverage and independent negative tests
# A deliberate mismatch must be rejected by the loader (no writes, no source changes).
def must_reject(fn,what):
    try:fn()
    except (ValueError,KeyError,AssertionError):return f'PASS — rejects {what}'
    raise AssertionError(f'Safety failure: accepted an invalid {what}')

mismatch=index.iloc[:1].copy();mismatch.loc[mismatch.index[0],'station_id']='a_nonexistent_station'
print(must_reject(lambda:weather_loader.for_station_days(mismatch),'incorrect station ID'))
mismatch=index.iloc[:1].copy();mismatch.loc[mismatch.index[0],'forecast_date']='2025-11-18'
print(must_reject(lambda:weather_loader.for_station_days(mismatch),'incorrect forecast date'))
print(must_reject(lambda:env_loader.get_vector('a_nonexistent_station'),'unknown environmental station'))

assert X_weather.shape[0]==X_environment.shape[0]==len(current)==len(y_next)==len(y_delta)==len(index)
assert (X_weather[:,:,0]>=-0.01).all(),'Negative daily precipitation below accepted tolerance'
assert ((X_weather[:,:,1]>=X_weather[:,:,2]-1e-3)).all(),'Daily temperature lower than dewpoint — review'
assert (X_weather[:,:,3]>400).all() and (X_weather[:,:,3]<1100).all(),'Unrealistic pressure'
assert (X_environment[:,3:8]>=0).all() and (X_environment[:,3:8]<=1).all(),'Invalid environment fractions'

row_flags=pd.DataFrame({
  'record_id':index['record_id'].astype(str),
  'station_id':index['station_id'].astype(str),
  'forecast_date':index['forecast_date'].astype(str),
  'split':index['split'].astype(str),
  'weather_complete_14x4':np.isfinite(X_weather).all(axis=(1,2)),
  'environment_complete_9':np.isfinite(X_environment).all(axis=1),
  'current_stage_finite':np.isfinite(current[:,0]),
  'next_stage_finite':np.isfinite(y_next[:,0]),
  'target_arithmetic_valid':np.isclose(y_next[:,0]-current[:,0],y_delta[:,0],atol=1e-4),
  'origin_before_target':(next_target_time-origin).eq(pd.Timedelta(hours=24)).to_numpy(),
  'weather_last_hour_before_origin':(origin-last_hour_local).eq(pd.Timedelta(minutes=30)).to_numpy(),
  'candidate_polygon_qa_status':index['catchment_qa_status'].astype(str),
})
quality_columns=['weather_complete_14x4','environment_complete_9','current_stage_finite','next_stage_finite',
                 'target_arithmetic_valid','origin_before_target','weather_last_hour_before_origin']
assert row_flags[quality_columns].all().all(),'Some station-day QA flag FAILED'

split_report=index.groupby('split',sort=False).agg(n_records=('record_id','count'),gauges=('station_id','nunique'),
                                                   min_date=('forecast_date','min'),max_date=('forecast_date','max')).reset_index()
feature_stats=pd.DataFrame({'feature':list(WEATHER_COLUMNS),
                            'min':X_weather.min(axis=(0,1)),
                            'mean':X_weather.mean(axis=(0,1)),
                            'max':X_weather.max(axis=(0,1))})
print('PASS: every station-day passes all seven row-level QA checks; no missing or out-of-order samples.')
print('PASS: negative-test guards reject incorrect station/date IDs.')
print('Split summary:');display(split_report)
print('Weather value audit (native unscaled units):');display(feature_stats.round(4))
env_by_station=env_loader.get_batch(sorted(index['station_id'].unique()))
env_stats=pd.DataFrame({'feature':list(ENV_COLUMNS),
                        'min':env_by_station.min(axis=0),
                        'mean':env_by_station.mean(axis=0),
                        'max':env_by_station.max(axis=0)})
print('Environmental value audit (39 unique labelled catchments, not 1429 repeated rows):')
display(env_stats.round(5))
print('Water level stage audit (metres; negative levels are not automatically errors):')
display(pd.DataFrame({'quantity':['current stage','next-day stage','observed 24h change'],
                      'min':[float(current.min()),float(y_next.min()),float(y_delta.min())],
                      'max':[float(current.max()),float(y_next.max()),float(y_delta.max())]}).round(4))
print('QA flag counts (sample-days):');display(row_flags['candidate_polygon_qa_status'].value_counts().to_frame('samples'))

PASS — rejects incorrect station ID
PASS — rejects incorrect forecast date
PASS — rejects unknown environmental station
PASS: every station-day passes all seven row-level QA checks; no missing or out-of-order samples.
PASS: negative-test guards reject incorrect station/date IDs.
Split summary:


,split,n_records,gauges,min_date,max_date
0,TRAIN,809,39,2025-11-17,2025-12-09
1,VALIDATION,269,39,2025-12-10,2025-12-16
2,TEST,351,39,2025-12-17,2025-12-25


Weather value audit (native unscaled units):


,feature,min,mean,max
0,precipitation_mm,0.000400,17.279699,227.409805
1,temperature_C,17.427299,23.278200,27.938400
2,dewpoint_C,13.247900,21.060699,23.676800
3,surface_pressure_hPa,863.075378,965.290894,1000.654785


Environmental value audit (39 unique labelled catchments, not 1429 repeated rows):


,feature,min,mean,max
0,env_log1p_area_km2,4.342890,6.32493,8.944230
1,env_mean_elevation_m,103.254257,435.92569,1564.021240
2,env_mean_slope_deg,2.462910,13.36926,20.352501
3,env_tree_fraction,0.566640,0.86002,0.995280
4,env_cropland_fraction,0.000330,0.04302,0.189950
5,env_built_fraction,0.000520,0.00720,0.049530
6,env_worldcover_water_fraction,0.000000,0.00575,0.024370
7,env_jrc_persistent_water_fraction,0.000000,0.00373,0.018560
8,env_hydrorivers_density_km_per_km2,0.236490,0.33697,0.434490


Water level stage audit (metres; negative levels are not automatically errors):


,quantity,min,max
0,current stage,-2.40,21.39
1,next-day stage,-1.95,21.39
2,observed 24h change,-4.96,5.18


QA flag counts (sample-days):


,samples
candidate_polygon_qa_status,
CANDIDATE_PENDING_VISUAL_QA,1291
REVIEW_OFFICIAL_DRAINAGE_DIFFERENCE_GT20,78
REVIEW_DISTANT_OUTLET,36
REVIEW_SOURCE_COORDINATE_DISCREPANCY,24


In [8]:
import importlib
import c4_fusion.src.satellite.sentinel1.load as s1_module

importlib.reload(s1_module)
Sentinel1PatchLoader = s1_module.Sentinel1PatchLoader

In [ ]:
# Cell 7 — Read recovered Sentinel-1 index with STRICT Glencourse-aware loader
from c4_fusion.src.satellite.sentinel1.load import Sentinel1PatchLoader

SAR_DIR=ROOT/'data'/'processed'/'c4'/'satellite'/'sentinel1'
SAR_EXTENSION_DIR=SAR_DIR/'glencourse_mosaic_extension'
SAR_INDEX_PATH=SAR_EXTENSION_DIR/'c4_s1_sample_index_1429_GLENC_MOSAIC_PRIVATE.csv'
SAR_QA_PATH=SAR_EXTENSION_DIR/'c4_s1_glencourse_mosaic_QA_PRIVATE.csv'
SAR_MANIFEST_PATH=SAR_EXTENSION_DIR/'c4_s1_glencourse_mosaic_manifest_PRIVATE.json'
ASCAT_DIR=ROOT/'data'/'processed'/'c4'/'satellite'/'ascat'
ASCAT_INDEX_PATH=ASCAT_DIR/'c4_ascat_swi_sample_index_1429_PRIVATE.csv'
ASCAT_QA_PATH=ASCAT_DIR/'c4_ascat_swi_row_QA_1429_PRIVATE.csv'

for p in (SAR_INDEX_PATH,SAR_QA_PATH,SAR_MANIFEST_PATH,ASCAT_INDEX_PATH,ASCAT_QA_PATH):
    if not p.is_file():
        raise FileNotFoundError(f'Required satellite output missing: {p}. Do not rerun Earth Engine; move the existing exported file into the documented folder.')

sar_loader=Sentinel1PatchLoader(repo_root=ROOT, data_dir=SAR_DIR, verify_provenance=True,
                               verify_patch_hashes=True)
assert sar_loader.index_path.resolve()==SAR_INDEX_PATH.resolve(), 'Wrong Sentinel-1 mapping (Glencourse overlay not selected)!'
sar_idx=sar_loader.table()
assert sar_loader.sample_count==EXPECTED_N and sar_loader.station_count==EXPECTED_GAUGES
assert len(sar_idx)==len(index)
identity_cols=('record_id','station_id','forecast_date','split')
for col in identity_cols:
    assert sar_idx[col].astype(str).reset_index(drop=True).equals(index[col].astype(str).reset_index(drop=True)),f'SAR misalignment in {col}'
assert sar_idx.record_id.is_unique
assert sar_idx.sar_available.dtype==bool
assert sar_idx.loc[sar_idx.sar_available,'sar_age_hours'].between(0,18*24,inclusive='right').all()
assert sar_idx.loc[sar_idx.sar_available,'sar_valid_pixel_fraction'].between(0.97,1.0).all()
assert (sar_idx['sar_validity_flag'].astype(bool)==sar_idx['sar_available']).all()

s1_mosaic_rows=sar_idx['sar_scene_id'].astype(str).str.startswith('GLEN_S1_')
assert int(s1_mosaic_rows.sum())==21 and set(sar_idx.loc[s1_mosaic_rows,'station_id'])=={'glencourse'}
assert sar_idx.loc[s1_mosaic_rows,'sar_scene_id'].nunique()==3
assert sar_idx.loc[s1_mosaic_rows,'sar_valid_pixel_fraction'].gt(0.97).all()

print('PASS: STRICT Sentinel-1 Glencourse extension manifest, file hashes and 1408-row preservation verified.')
print('Path:',SAR_INDEX_PATH)
print('SAR available:',int(sar_idx.sar_available.sum()),'/',EXPECTED_N)
print('Glencourse recovered:',int(s1_mosaic_rows.sum()),'rows from',sar_idx.loc[s1_mosaic_rows,'sar_scene_id'].nunique(),'mosaics')

In [10]:
# Cell 8 — Read ASCAT SWI index + per-row QA; align on frozen record identity/order
swi_idx=pd.read_csv(ASCAT_INDEX_PATH,encoding='utf-8-sig',low_memory=False)
swi_qa=pd.read_csv(ASCAT_QA_PATH,encoding='utf-8-sig',low_memory=False)
SWI_FEATURE_COLUMNS=('swi001_percent','swi005_percent','swi010_percent')
SWI_QFLAG_COLUMNS=('qflag001_percent','qflag005_percent','qflag010_percent')
SWI_QA_COLUMNS=('id_match','split_match','validity_consistent','pre_origin_source',
                'feature_complete_when_available','missing_unfabricated','qa_pass')
required_swi=set(identity_cols) | {'swi_available','swi_source_date','swi_age_days','swi_age_logscaled',
    'swi_validity_flag','extraction_method','area_valid_fraction','nearest_distance_km'} | set(SWI_FEATURE_COLUMNS) | set(SWI_QFLAG_COLUMNS)
assert required_swi.issubset(swi_idx.columns),f'ASCAT missing columns: {required_swi-set(swi_idx.columns)}'
assert (set(identity_cols)|set(SWI_QA_COLUMNS)|{'swi_available','swi_source_date','swi_age_days'}).issubset(swi_qa.columns)
assert len(swi_idx)==len(swi_qa)==EXPECTED_N and swi_idx.record_id.is_unique and swi_qa.record_id.is_unique
for source_name,frame in [('ASCAT index',swi_idx),('ASCAT QA',swi_qa)]:
    for col in identity_cols:
        assert frame[col].astype(str).reset_index(drop=True).equals(index[col].astype(str).reset_index(drop=True)),f'{source_name} alignment failed: {col}'
for col in ('swi_available','swi_source_date','swi_age_days'):
    assert swi_idx[col].astype(str).equals(swi_qa[col].astype(str)),f'ASCAT source versus QA disagrees: {col}'
assert swi_idx.swi_available.dtype==bool and swi_qa.qa_pass.dtype==bool
assert swi_idx.swi_validity_flag.isin([0,1]).all()
assert swi_idx.swi_validity_flag.astype(bool).eq(swi_idx.swi_available).all()
assert swi_idx['extraction_method'].notna().all()

swi_available=swi_idx['swi_available'].to_numpy(dtype=bool)
swi_source=pd.to_datetime(swi_idx['swi_source_date'],errors='coerce')
swi_age=pd.to_numeric(swi_idx['swi_age_days'],errors='coerce')
swi_scaled=pd.to_numeric(swi_idx['swi_age_logscaled'],errors='coerce')
assert swi_source[swi_available].notna().all() and (swi_source[swi_available]<forecast_dt[swi_available]).all(), 'ASCAT look-ahead!'
assert np.allclose((forecast_dt[swi_available]-swi_source[swi_available]).dt.days.to_numpy(),swi_age[swi_available],rtol=0,atol=0)
assert (swi_age[swi_available]>0).all()
assert np.isfinite(swi_scaled[swi_available]).all()
assert np.allclose(swi_scaled[swi_available],np.log1p(swi_age[swi_available])/np.log1p(7.0),rtol=0,atol=1e-6), 'ASCAT 7-day age scaling changed'
assert all(swi_qa[c].dtype==bool for c in SWI_QA_COLUMNS)

X_ascat_swi=swi_idx[list(SWI_FEATURE_COLUMNS)].to_numpy(dtype=np.float32,copy=True)
X_ascat_qflag_percent=swi_idx[list(SWI_QFLAG_COLUMNS)].to_numpy(dtype=np.float32,copy=True)
assert X_ascat_swi.shape==(EXPECTED_N,3) and X_ascat_qflag_percent.shape==(EXPECTED_N,3)
swi_features_finite=np.isfinite(X_ascat_swi).all(axis=1)
swi_qflags_finite=np.isfinite(X_ascat_qflag_percent).all(axis=1)
assert swi_features_finite[swi_available].all(), 'Available ASCAT SWI has missing feature(s)'
assert swi_qflags_finite[swi_available].all(), 'Available ASCAT quality indicators missing'
assert np.logical_and(X_ascat_swi[swi_available]>=0,X_ascat_swi[swi_available]<=100).all(), 'SWI percent out of range'
assert np.logical_and(X_ascat_qflag_percent[swi_available]>=0,X_ascat_qflag_percent[swi_available]<=100).all(), 'ASCAT qflag percent out of range'
assert np.isfinite(swi_idx.loc[swi_idx['swi_available'],'area_valid_fraction']).all() and (swi_idx.loc[swi_idx['swi_available'],'area_valid_fraction']>=-1e-9).all() and (swi_idx.loc[swi_idx['swi_available'],'area_valid_fraction']<=1+1e-9).all(), 'ASCAT area coverage outside 0..1 (allowing float roundoff)' 
assert all(swi_idx.loc[~swi_idx['swi_available'],list(SWI_FEATURE_COLUMNS)].isna().all())

print('PASS: ASCAT index and row-QA identity/order, date and age policy validated.')
print('ASCAT SWI available:',int(swi_available.sum()),'/',EXPECTED_N,'; row-QA passing:',int(swi_qa.qa_pass.sum()))
print('ASCAT SWI features:',SWI_FEATURE_COLUMNS,'quality percentages:',SWI_QFLAG_COLUMNS)
print('Note: ASCAT area_valid_fraction is not Sentinel-1 image pixel-coverage; their thresholds must not be conflated.')


PASS: ASCAT index and row-QA identity/order, date and age policy validated.
ASCAT SWI available: 1429 / 1429 ; row-QA passing: 1429
ASCAT SWI features: ('swi001_percent', 'swi005_percent', 'swi010_percent') quality percentages: ('qflag001_percent', 'qflag005_percent', 'qflag010_percent')
Note: ASCAT area_valid_fraction is not Sentinel-1 image pixel-coverage; their thresholds must not be conflated.


In [ ]:
# Cell 9 — On-disk SAR pixel/hash verification: all 264 UNIQUE NPZ patches, no Earth Engine
sar_patch_verification=sar_loader.verify_all_patches()
assert sar_patch_verification['available']==int(sar_idx.sar_available.sum())
assert sar_patch_verification['unique_patches_verified']==264
assert sar_patch_verification['original']==261 and sar_patch_verification['glencourse_mosaics']==3
SAR_PATCHES_VERIFIED=True
print('PASS: all original 261 + 3 new Glencourse NPZ files physically verified, including SHA256, pixel masks and VV/VH values.')
print(sar_patch_verification)

# A real recovered sample confirms the lazily accessed tensor and age/availability contract.
probe=sar_loader.get_sample('glencourse_2025-11-17')
assert probe['sar_image'].shape==(256,256,2) and probe['sar_valid_mask'].shape==(256,256)
assert int(probe['sar_available'])==1 and np.isfinite(probe['sar_age_hours'])
assert probe['sar_valid_mask'].mean()>0.97
print('Glencourse test sample:',probe['record_id'],'available',probe['sar_available'],
      'age (hours)',float(probe['sar_age_hours']),'valid pixels',float(probe['sar_valid_mask'].mean()))


PASS: all original 261 + 3 new Glencourse NPZ files physically verified, including SHA256, pixel masks and VV/VH values.
{'station_days': 1429, 'available': 1429, 'unique_patches_verified': 264, 'original': 261, 'glencourse_mosaics': 3}
Glencourse test sample: glencourse_2025-11-17 available 1.0 age (hours) 75.07917022705078 valid pixels 0.9999237060546875


In [12]:
# Cell 10 — Final row-level JOINT QA and ACTUAL complete-sample accounting
# Retain original seven weather/environment/target checks; extend them with independent satellite checks.
assert SAR_PATCHES_VERIFIED is True, 'Run Cell 9 and verify actual NPZ patches first.'
assert row_flags['record_id'].astype(str).equals(index['record_id'].astype(str))

row_flags['sar_available']=sar_idx['sar_available'].to_numpy(dtype=bool)
row_flags['sar_age_within_18_days']=pd.to_numeric(sar_idx['sar_age_hours'],errors='coerce').between(0,18*24,inclusive='right').to_numpy()
row_flags['sar_joint_pixel_coverage_at_least_97pct']=pd.to_numeric(sar_idx['sar_valid_pixel_fraction'],errors='coerce').between(.97,1.0).to_numpy()
row_flags['sar_cached_patch_verified']=row_flags['sar_available'] & SAR_PATCHES_VERIFIED
row_flags['sar_glencourse_mosaic']=s1_mosaic_rows.to_numpy(dtype=bool)
row_flags['swi_available']=swi_available
row_flags['swi_features_complete_3']=swi_features_finite
row_flags['swi_qflags_complete_3']=swi_qflags_finite
row_flags['swi_source_before_origin']=swi_source.lt(forecast_dt).fillna(False).to_numpy(dtype=bool)
row_flags['swi_row_QA_pass']=swi_qa['qa_pass'].to_numpy(dtype=bool)
row_flags['swi_age_valid']=((swi_age>0)&np.isfinite(swi_age)&np.isfinite(swi_scaled)).to_numpy(dtype=bool)

weather_env_flags=('weather_complete_14x4','environment_complete_9','weather_last_hour_before_origin')
sar_flags=('sar_available','sar_age_within_18_days','sar_joint_pixel_coverage_at_least_97pct','sar_cached_patch_verified')
swi_flags=('swi_available','swi_features_complete_3','swi_qflags_complete_3',
           'swi_source_before_origin','swi_age_valid','swi_row_QA_pass')
input_flags=(*weather_env_flags,'current_stage_finite',*sar_flags,*swi_flags)
supervised_flags=(*input_flags,'next_stage_finite','target_arithmetic_valid','origin_before_target')
row_flags['feature_complete']=row_flags[list(input_flags)].all(axis=1)
row_flags['supervised_complete']=row_flags[list(supervised_flags)].all(axis=1)
row_flags['incomplete_reason']=row_flags.apply(
    lambda r:'; '.join(col for col in supervised_flags if not bool(r[col])),axis=1)

n_feature_complete=int(row_flags.feature_complete.sum())
n_supervised_complete=int(row_flags.supervised_complete.sum())
assert n_supervised_complete<=n_feature_complete<=EXPECTED_N
assert (row_flags['candidate_polygon_qa_status'].astype(str)==index['catchment_qa_status'].astype(str)).all()
assert not BANNED_INPUT_FIELDS.intersection(sar_idx.columns) and not BANNED_INPUT_FIELDS.intersection(swi_idx.columns)

report_by_split=(row_flags.groupby('split',sort=False)
    .agg(n_records=('record_id','size'),weather_complete=('weather_complete_14x4','sum'),
         environment_complete=('environment_complete_9','sum'),sar_available=('sar_available','sum'),
         swi_available=('swi_available','sum'),feature_complete=('feature_complete','sum'),
         supervised_complete=('supervised_complete','sum')).reset_index())

print('ACTUAL FEATURE-COMPLETE:',n_feature_complete,'/',EXPECTED_N)
print('ACTUAL SUPERVISED-COMPLETE:',n_supervised_complete,'/',EXPECTED_N)
print('Split-level counts:')
display(report_by_split)
if n_supervised_complete<EXPECTED_N:
    print('REVIEW: excluded rows and exact failing flags:')
    display(row_flags.loc[~row_flags.supervised_complete,['record_id','split','incomplete_reason']].head(50))
else:
    print('PASS: All 1429 station-days are supervised-complete across weather, environment, Sentinel-1 and ASCAT.')
print('Geometry note: candidate catchment QA statuses stay provisional; completion does not mean geometry independently certified.')


ACTUAL FEATURE-COMPLETE: 1429 / 1429
ACTUAL SUPERVISED-COMPLETE: 1429 / 1429
Split-level counts:


,split,n_records,weather_complete,environment_complete,sar_available,swi_available,feature_complete,supervised_complete
0,TRAIN,809,809,809,809,809,809,809
1,VALIDATION,269,269,269,269,269,269,269
2,TEST,351,351,351,351,351,351,351


PASS: All 1429 station-days are supervised-complete across weather, environment, Sentinel-1 and ASCAT.
Geometry note: candidate catchment QA statuses stay provisional; completion does not mean geometry independently certified.


In [ ]:
# Cell 11 — Atomic/idempotent FINAL export: aligned inputs, separate labels, index, QA and manifest
# Does NOT duplicate 1429 SAR images. SAR is resolved lazily using NPZ references + existing load.py.
# The original Notebook 06 preliminary files (c4_preliminary_M6_*) are never overwritten.
def npz_bytes(**arrays)->bytes:
    mem=io.BytesIO()
    np.savez_compressed(mem,**arrays)
    return mem.getvalue()

def save_if_changed(path:Path,content:bytes)->str:
    path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists() and path.read_bytes()==content:
        return 'UNCHANGED'
    temporary=path.with_name(path.name+'.writing.tmp')
    try:
        temporary.write_bytes(content)
        os.replace(temporary,path)
    finally:
        if temporary.exists():temporary.unlink()
    return 'WRITTEN'

assert SAR_PATCHES_VERIFIED and len(row_flags)==EXPECTED_N
assert row_flags['feature_complete'].sum()==n_feature_complete
assert row_flags['supervised_complete'].sum()==n_supervised_complete

final_index_cols=['record_id','station_id','forecast_date','split',
    'hindcast_origin_0900_local','target_time_0900_local','catchment_qa_status']
aligned_index=index[final_index_cols].copy()
aligned_index.insert(0,'array_row',np.arange(EXPECTED_N,dtype=np.int32))
aligned_index['weather_record_id']=aligned_index['record_id']
aligned_index['environment_station_id']=aligned_index['station_id']

# Satellite provenance + age/availability; storage location relative to repository root.
for col in ('sar_available','sar_status','sar_scene_id','sar_acquisition_utc','sar_age_hours',
            'sar_age_logscaled','sar_validity_flag','sar_valid_pixel_fraction','sar_orbit_pass',
            'sar_relative_orbit','patch_relpath','patch_sha256'):
    aligned_index[col]=sar_idx[col].to_numpy(copy=True)
for col in ('swi_available','swi_source_date','swi_age_days','swi_age_logscaled',
            'swi_validity_flag','extraction_method','area_valid_fraction','nearest_distance_km'):
    aligned_index[col]=swi_idx[col].to_numpy(copy=True)
aligned_index['feature_complete']=row_flags['feature_complete'].to_numpy(dtype=bool)
aligned_index['supervised_complete']=row_flags['supervised_complete'].to_numpy(dtype=bool)
assert len(aligned_index)==EXPECTED_N and aligned_index['record_id'].tolist()==index['record_id'].tolist()
assert not BANNED_INPUT_FIELDS.intersection(aligned_index.columns)

input_payload=npz_bytes(
    X_weather=X_weather.astype(np.float32,copy=False),
    X_environment=X_environment.astype(np.float32,copy=False),
    X_ascat_swi=X_ascat_swi.astype(np.float32,copy=False),
    # Quality flags are auxiliary QA data, NOT additional SWI soil-moisture channels.
    X_ascat_qflag_percent=X_ascat_qflag_percent.astype(np.float32,copy=False),
    stage_current_m=current.astype(np.float32,copy=False),
    sar_available=sar_idx['sar_available'].to_numpy(dtype=np.float32).reshape(-1,1),
    sar_age_hours=sar_idx['sar_age_hours'].to_numpy(dtype=np.float32).reshape(-1,1),
    sar_age_logscaled=sar_idx['sar_age_logscaled'].to_numpy(dtype=np.float32).reshape(-1,1),
    sar_valid_pixel_fraction=sar_idx['sar_valid_pixel_fraction'].to_numpy(dtype=np.float32).reshape(-1,1),
    swi_available=swi_idx['swi_available'].to_numpy(dtype=np.float32).reshape(-1,1),
    swi_age_days=swi_idx['swi_age_days'].to_numpy(dtype=np.float32).reshape(-1,1),
    swi_age_logscaled=swi_idx['swi_age_logscaled'].to_numpy(dtype=np.float32).reshape(-1,1),
    swi_area_valid_fraction=swi_idx['area_valid_fraction'].to_numpy(dtype=np.float32).reshape(-1,1),
    feature_complete=row_flags['feature_complete'].to_numpy(dtype=np.uint8).reshape(-1,1),
    record_id=index['record_id'].astype(str).to_numpy(dtype=str),
    station_id=index['station_id'].astype(str).to_numpy(dtype=str),
    weather_feature_names=np.asarray(WEATHER_COLUMNS,dtype=str),
    environment_feature_names=np.asarray(ENV_COLUMNS,dtype=str),
    ascat_swi_feature_names=np.asarray(SWI_FEATURE_COLUMNS,dtype=str),
    ascat_qflag_names=np.asarray(SWI_QFLAG_COLUMNS,dtype=str),
)
label_payload=npz_bytes(
    y_delta_stage_m=y_delta.astype(np.float32,copy=False),
    y_next_stage_m=y_next.astype(np.float32,copy=False),
    record_id=index['record_id'].astype(str).to_numpy(dtype=str),
)

# Hash and count provenance saved alongside final outputs. This report is calculated only
# after physical NPZ validation and checks of all 1429 source identities.
source_files={'notebook01_target_free':INDEX_PATH,'notebook01_private_labels':LABELS_PATH,
              'notebook01_spatial_QA':SPATIAL_QA_PATH,
              'sentinel1_glencourse_index':SAR_INDEX_PATH,
              'sentinel1_glencourse_manifest':SAR_MANIFEST_PATH,
              'sentinel1_glencourse_QA':SAR_QA_PATH,
              'ascat_index':ASCAT_INDEX_PATH,'ascat_row_QA':ASCAT_QA_PATH}
summary={
    'title':'C4 final M6 multimodal alignment QA',
    'status':'PASS_PHYSICAL_SAR_PATCHES_VALIDATED',
    'expected_station_days':EXPECTED_N,
    'station_count':EXPECTED_GAUGES,
    'actual_feature_complete':n_feature_complete,
    'actual_supervised_complete':n_supervised_complete,
    'actual_feature_complete_fraction':n_feature_complete/EXPECTED_N,
    'actual_supervised_complete_fraction':n_supervised_complete/EXPECTED_N,
    'per_split':report_by_split.to_dict(orient='records'),
    'available_per_source':{
        'weather':int(row_flags.weather_complete_14x4.sum()),
        'environment':int(row_flags.environment_complete_9.sum()),
        'sentinel1':int(row_flags.sar_available.sum()),
        'ascat':int(row_flags.swi_available.sum()),
    },
    'sar_verified_patches':sar_patch_verification,
    'glencourse_recovered_station_days':int(s1_mosaic_rows.sum()),
    'glencourse_new_unique_mosaics':int(sar_idx.loc[s1_mosaic_rows,'sar_scene_id'].nunique()),
    'swi_feature_columns':list(SWI_FEATURE_COLUMNS),
    'swi_quality_columns':list(SWI_QFLAG_COLUMNS),
    's1_index_repo_relpath':SAR_INDEX_PATH.relative_to(ROOT).as_posix(),
    'ascat_index_repo_relpath':ASCAT_INDEX_PATH.relative_to(ROOT).as_posix(),
    'source_file_sha256':{k:sha256(p) for k,p in source_files.items()},
    'limitations':['SAR 50m grid is not native Sentinel-1 spatial resolution',
                   'SAR scene reused on later forecast days is not an independent acquisition',
                   'Retrospective 09:00 hindcast; operational publication latency unverified',
                   'Catchment geometry is candidate/pending QA, not independently verified',
                   'ASCAT quality percentages are auxiliary QA, not SWI percentage channels',
                   'SAR VV/VH tensors remain in per-scene NPZ cache and are loaded lazily'],
}
final_outputs={
  'c4_aligned_inputs_1429_PRIVATE.npz':input_payload,
  'c4_aligned_labels_1429_PRIVATE.npz':label_payload,
  'c4_aligned_index_1429_PRIVATE.csv':aligned_index.to_csv(index=False).encode('utf-8-sig'),
  'c4_aligned_row_QA_1429_PRIVATE.csv':row_flags.to_csv(index=False).encode('utf-8-sig'),
  'c4_aligned_summary_PRIVATE.json':(json.dumps(summary,indent=2,ensure_ascii=False,allow_nan=False)+'\n').encode('utf-8'),
}

for name,payload in final_outputs.items():
    print(name+':',save_if_changed(OUT_DIR/name,payload))
# Reopen inputs+labels to check the saved format is structurally aligned.
with np.load(OUT_DIR/'c4_aligned_inputs_1429_PRIVATE.npz',allow_pickle=False) as saved:
    assert saved['X_weather'].shape==EXPECTED_WEATHER_SHAPE
    assert saved['X_environment'].shape==EXPECTED_ENV_SHAPE
    assert saved['X_ascat_swi'].shape==(EXPECTED_N,3)
    assert saved['record_id'].tolist()==index['record_id'].astype(str).tolist()
with np.load(OUT_DIR/'c4_aligned_labels_1429_PRIVATE.npz',allow_pickle=False) as saved_labels:
    assert saved_labels['record_id'].tolist()==index['record_id'].astype(str).tolist()
assert json.loads((OUT_DIR/'c4_aligned_summary_PRIVATE.json').read_text(encoding='utf-8'))['actual_supervised_complete']==n_supervised_complete
print(f'FINAL: {n_supervised_complete}/{EXPECTED_N} supervised-complete; {n_feature_complete}/{EXPECTED_N} feature-complete. Zero original inputs changed.')
print('Saved:',OUT_DIR)
